In [0]:
%run ./00_install

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.raw;
CREATE SCHEMA IF NOT EXISTS workspace.quality;
CREATE SCHEMA IF NOT EXISTS workspace.artifacts;
CREATE VOLUME IF NOT EXISTS workspace.raw.retail_lab;
CREATE VOLUME IF NOT EXISTS workspace.bronze.retail_lab;
CREATE VOLUME IF NOT EXISTS workspace.silver.retail_lab;
CREATE VOLUME IF NOT EXISTS workspace.gold.retail_lab;
CREATE VOLUME IF NOT EXISTS workspace.quality.retail_lab;
CREATE VOLUME IF NOT EXISTS workspace.artifacts.retail_lab;

In [0]:
from pathlib import Path
from pydantic import BaseModel, Field
from datetime import date
import uuid

# Definimos una clase que hereda de BaseModel de Pydantic.
# Esto nos permite validar automáticamente los tipos de datos y valores.
class LabSettings(BaseModel):
    # Definición de parámetros básicos con valores por defecto (Default)
    catalog: str = "workspace"
    schema_name_raw: str = "raw"
    schema_name_bronze: str = "bronze"
    schema_name_silver: str = "silver"
    schema_name_quality: str = "quality"
    schema_name_gold: str = "gold"
    schema_name_artifacts: str = "artifacts"
    volume_name: str = "retail_lab"
    
    # Campo con validación: debe ser un string con formato de fecha (AAAA-MM-DD)
    # Por defecto toma la fecha de hoy.
    run_date: str = Field(default=str(date.today()), pattern=r"^\d{4}-\d{2}-\d{2}$")

    # Parámetros numéricos con restricciones de valor mínimo (ge = greater than or equal)
    n_customers: int = Field(default=5_000, ge=100) # Mínimo 100 clientes
    n_products: int = Field(default=500, ge=10)     # Mínimo 10 productos
    n_orders: int = Field(default=50_000, ge=1_000) # Mínimo 1,000 órdenes

    # Generación dinámica: crea un ID de lote único usando la librería uuid.
    # default_factory permite ejecutar una función cada vez que se crea una instancia.
    batch_id: str = Field(default_factory=lambda: uuid.uuid4().hex[:12])

    # Propiedad calculada: no es un dato que se guarde, sino que se genera
    # a partir de los otros valores para dar la ruta de almacenamiento.
    @property
    def raw(self) -> Path:
        # Devuelve una ruta de sistema de archivos usando Path de pathlib
        return Path(f"/Volumes/{self.catalog}/{self.schema_name_raw}/{self.volume_name}")

    @property
    def bronze(self) -> Path:
        # Devuelve una ruta de sistema de archivos usando Path de pathlib
        return Path(f"/Volumes/{self.catalog}/{self.schema_name_bronze}/{self.volume_name}")
    
    @property
    def silver(self) -> Path:
        # Devuelve una ruta de sistema de archivos usando Path de pathlib
        return Path(f"/Volumes/{self.catalog}/{self.schema_name_silver}/{self.volume_name}")
    
    @property
    def gold(self) -> Path:
        # Devuelve una ruta de sistema de archivos usando Path de pathlib
        return Path(f"/Volumes/{self.catalog}/{self.schema_name_gold}/{self.volume_name}")
    
    @property
    def quality(self) -> Path:
        # Devuelve una ruta de sistema de archivos usando Path de pathlib
        return Path(f"/Volumes/{self.catalog}/{self.schema_name_quality}/{self.volume_name}")
    
    @property
    def artifacts(self) -> Path:
        # Devuelve una ruta de sistema de archivos usando Path de pathlib
        return Path(f"/Volumes/{self.catalog}/{self.schema_name_artifacts}/{self.volume_name}")
    
    
# Instanciamos la configuración. Pydantic valida todo al crear este objeto.
settings = LabSettings()

In [0]:
# model_dump() convierte el objeto en un diccionario de Python estándar.
print(settings.model_dump())

In [0]:
# Accedemos a la propiedad calculada volume_root.
print("Root:", settings.raw)
print("Root:", settings.bronze)
print("Root:", settings.silver)
print("Root:", settings.gold)
print("Root:", settings.gold)
print("Root:", settings.quality)
print("Root:", settings.artifacts)